# Statement Embedding with Haiku Model Analysis

This notebook demonstrates:
- Converting a statement into embedding using Sentence Transformer
- Sending embedding information to Claude Haiku model for analysis
- Displaying both embeddings and model response

## 1. Install Dependencies

In [ ]:
# Install required libraries
!pip install sentence-transformers anthropic numpy python-dotenv -q

## 2. Import Libraries

In [ ]:
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic
import numpy as np
import os
import json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("All libraries imported successfully!")

## 3. Set Up API Key

In [ ]:
# Debug: Check if .env file is being loaded correctly
import os
from pathlib import Path

print("Debugging API Key Loading:")
print("="*70)

# Check current working directory
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# Check if .env file exists
env_file = Path(cwd) / ".env"
print(f"\n.env file path: {env_file}")
print(f".env file exists: {env_file.exists()}")

if env_file.exists():
    print("\n.env file contents (first 100 chars):")
    with open(env_file, 'r') as f:
        content = f.read()
        print(f"  {content[:100]}...")
        
# Check environment variables
api_key = os.getenv('ANTHROPIC_API_KEY')
print(f"\nAPI Key loaded: {bool(api_key)}")
if api_key:
    print(f"API Key format: {api_key[:20]}...{api_key[-10:]}")
    print(f"API Key length: {len(api_key)}")
    print(f"Has 'sk-ant' prefix: {api_key.startswith('sk-ant')}")
else:
    print("❌ API Key NOT found in environment!")
    print("\nTry these fixes:")
    print("1. Make sure .env is in: /home/ubuntu/AI_Model/.env")
    print("2. Format should be: ANTHROPIC_API_KEY=sk-ant-xxxxx")
    print("3. No spaces, no quotes around the key")
    print("4. Restart Jupyter kernel after fixing .env")

In [ ]:
# Load API key from .env file
api_key = os.getenv('ANTHROPIC_API_KEY')

if not api_key:
    raise ValueError("ANTHROPIC_API_KEY not found! Make sure it's in your .env file as: ANTHROPIC_API_KEY=sk-ant-...")

# Initialize the Anthropic client
client = Anthropic(api_key=api_key)
print(f"✅ API Key loaded from .env file successfully!")
print(f"✅ Using model: claude-haiku-4-5-20251001")

## 4. Load Sentence Transformer Model

In [ ]:
print("Loading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded successfully!")
print(f"Model dimension: {model.get_sentence_embedding_dimension()}")

## 5. Define Statement

In [ ]:
# Define the statement to convert into embedding
statement = "Artificial intelligence is revolutionizing how we solve complex problems and make decisions."

print("="*70)
print("STATEMENT TO ANALYZE")
print("="*70)
print(f"\n{statement}")
print(f"\nStatement length: {len(statement)} characters")

## 6. Generate Embedding

In [ ]:
print("\nGenerating embedding...")
embedding = model.encode(statement, convert_to_tensor=False)

print(f"Embedding generated successfully!")
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding dtype: {embedding.dtype}")

## 7. Display First Five Dimensions

In [ ]:
print("="*70)
print("FIRST FIVE DIMENSIONS OF EMBEDDING")
print("="*70)

first_five = embedding[:5]
print(f"\nFirst 5 dimensions: {first_five}")

for i, val in enumerate(first_five):
    print(f"  Dimension {i+1}: {val:.6f}")

## 8. Display Full Embedding Statistics

In [ ]:
print("="*70)
print("EMBEDDING STATISTICS")
print("="*70)

print(f"\nTotal dimensions: {len(embedding)}")
print(f"Mean value: {np.mean(embedding):.6f}")
print(f"Std deviation: {np.std(embedding):.6f}")
print(f"Min value: {np.min(embedding):.6f}")
print(f"Max value: {np.max(embedding):.6f}")
print(f"L2 Norm: {np.linalg.norm(embedding):.6f}")

## 9. Prepare Embedding Data for Haiku Model

In [ ]:
# Prepare embedding summary for the model
embedding_summary = {
    "statement": statement,
    "embedding_dimension": len(embedding),
    "first_five_dimensions": first_five.tolist(),
    "embedding_statistics": {
        "mean": float(np.mean(embedding)),
        "std": float(np.std(embedding)),
        "min": float(np.min(embedding)),
        "max": float(np.max(embedding)),
        "l2_norm": float(np.linalg.norm(embedding))
    }
}

print("Embedding summary prepared:")
print(json.dumps(embedding_summary, indent=2))

## 10. Send to Haiku Model for Analysis

In [ ]:
# Prepare the prompt for Haiku
prompt = f"""I have converted the following statement into an embedding representation using a Sentence Transformer model.

Statement: "{statement}"

Embedding Information:
- Total dimensions: {len(embedding)}
- First 5 dimensions: {first_five.tolist()}
- Mean value: {np.mean(embedding):.6f}
- Std deviation: {np.std(embedding):.6f}
- Min value: {np.min(embedding):.6f}
- Max value: {np.max(embedding):.6f}
- L2 Norm: {np.linalg.norm(embedding):.6f}

Please provide:
1. An analysis of what this statement represents based on its semantic content
2. Key concepts and themes in the statement
3. What the embedding dimensions might capture about this text
4. Potential use cases for this embedding
5. Any insights about the text based on the embedding statistics
"""

print("Sending request to Haiku model...\n")

# Call the Haiku model
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

haiku_response = response.content[0].text
print("Response received from Haiku model!")

## 11. Display Haiku Model Response

In [ ]:
print("="*70)
print("HAIKU MODEL ANALYSIS AND RESPONSE")
print("="*70)
print()
print(haiku_response)

## 12. Display Model Usage Information

In [ ]:
print("="*70)
print("MODEL USAGE INFORMATION")
print("="*70)
print(f"\nModel: {response.model}")
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Total tokens: {response.usage.input_tokens + response.usage.output_tokens}")
print(f"Stop reason: {response.stop_reason}")

## 13. Interactive Analysis Function

In [ ]:
def analyze_statement_with_embedding(text):
    """
    Complete function to analyze any statement:
    1. Generate embedding
    2. Extract statistics
    3. Send to Haiku for analysis
    4. Display results
    """
    print(f"\n{'='*70}")
    print("ANALYZING NEW STATEMENT")
    print(f"{'='*70}\n")
    
    # Generate embedding
    print(f"Statement: {text}")
    embedding = model.encode(text, convert_to_tensor=False)
    first_five = embedding[:5]
    
    # Display embedding info
    print(f"\nFirst 5 dimensions: {first_five}")
    print(f"Embedding mean: {np.mean(embedding):.6f}")
    print(f"Embedding std: {np.std(embedding):.6f}")
    
    # Create prompt for Haiku
    prompt = f"""Analyze this statement and its embedding:
    
Statement: "{text}"

Embedding stats:
- Dimensions: {len(embedding)}
- First 5: {first_five.tolist()}
- Mean: {np.mean(embedding):.6f}
- Std: {np.std(embedding):.6f}

Provide a concise analysis focusing on semantic meaning and key themes.
"""
    
    # Get Haiku response
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    
    print(f"\nHaiku Response:\n")
    print(response.content[0].text)
    print(f"\nTokens used: {response.usage.input_tokens + response.usage.output_tokens}")

# Example usage
analyze_statement_with_embedding("Machine learning models learn patterns from data.")

## 14. Try Another Statement

In [ ]:
# Try analyzing another statement
analyze_statement_with_embedding("Natural language processing enables computers to understand human language.")

## 15. Summary

In [ ]:
print("="*70)
print("SUMMARY")
print("="*70)

summary = f"""
This notebook demonstrates:

1. EMBEDDING GENERATION:
   - Used Sentence Transformer model (all-MiniLM-L6-v2)
   - Converted statement into 384-dimensional vector representation
   - Extracted first 5 dimensions and statistical properties

2. HAIKU MODEL INTEGRATION:
   - Sent embedding information to Claude Haiku model
   - Haiku analyzed semantic content and embedding characteristics
   - Received AI-generated insights about the statement

3. KEY STATISTICS:
   - Embedding dimensions: {embedding.shape[0]}
   - First 5 values: {first_five}
   - Mean value: {np.mean(embedding):.6f}
   - L2 Norm: {np.linalg.norm(embedding):.6f}

4. USE CASES:
   - Semantic similarity comparisons
   - Information retrieval and ranking
   - Clustering similar documents
   - Transfer learning for NLP tasks
   - AI-powered content analysis

5. NEXT STEPS:
   - Modify the statement in Cell 5
   - Run cells 6-11 to analyze the new statement
   - Or use the interactive function in Cell 13
"""

print(summary)